In [ ]:
import kagglehub

path = kagglehub.dataset_download("altruistdelhite04/loan-prediction-problem-dataset")

print("Path to dataset files:", path)

In [ ]:
import os
os.listdir(path)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

import warnings
warnings.filterwarnings("ignore")

In [ ]:
train = pd.read_csv(os.path.join(path, "train_u6lujuX_CVtuZ9i.csv"))
print("Before removing duplicates:", train.shape)

train = train.drop_duplicates()

print("After removing duplicates:", train.shape)

The training dataset is loaded into a Pandas DataFrame.

The Loan_ID column is removed because it is an identifier and does not contribute to prediction.

In [ ]:
train.drop(columns='Loan_ID', inplace=True)

train.head()

In [ ]:
import numpy as np
import pandas as pd

numeric_cols = train.select_dtypes(include=np.number).columns

outlier_summary = []

for col in numeric_cols:
    Q1 = train[col].quantile(0.25)
    Q3 = train[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = train[(train[col] < lower) | (train[col] > upper)]

    outlier_summary.append({
        "Column": col,
        "Total Values": train.shape[0],
        "Outlier Count": outliers.shape[0],
        "Outlier %": round((outliers.shape[0] / train.shape[0]) * 100, 2)
    })

outlier_df = pd.DataFrame(outlier_summary)
outlier_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

for col in numeric_cols:
    plt.figure()
    sns.boxplot(x=train[col])
    plt.title(f"Boxplot of {col}")
    plt.show()

In [ ]:
!pip uninstall -y pandas-profiling
!pip install ydata-profiling

import pandas as pd
import numpy as np
from ydata_profiling import ProfileReport

prof = ProfileReport(train)
prof.to_file(output_file= "output.html")

Missing values are handled separately for categorical and numerical columns.

Categorical columns are filled using the mode (most frequent value).

Numerical columns are filled using the median to reduce the effect of outliers.

This approach preserves dataset size and prevents model bias.

In [ ]:
#. missing value handling
# changing Categorical → Mode
for col in train.select_dtypes(include='object').columns:
    train[col].fillna(train[col].mode()[0], inplace=True)

#changing Numerical → Median
for col in train.select_dtypes(include=np.number).columns:
    train[col].fillna(train[col].median(), inplace=True)

Machine learning models require numerical input. Therefore, categorical variables such as Gender, Married, Education, Self_Employed, and Property_Area are converted into numeric format using controlled manual mapping.

Binary categories are mapped to 0 and 1, ensuring interpretability and model compatibility.

In [ ]:
train.replace({
    'Married':{'No':0,'Yes':1},
    'Gender':{'Male':1,'Female':0},
    'Self_Employed':{'No':0,'Yes':1},
    'Property_Area':{'Rural':0,'Semiurban':1,'Urban':2},
    'Education':{'Graduate':1,'Not Graduate':0},
    'Loan_Status':{'N':0,'Y':1}
}, inplace=True)

The “Dependents” column contains a categorical value “3+” which represents three or more dependents.

This value is converted to numeric format (3) to ensure proper numerical processing and compatibility with machine learning models.

In [ ]:
train['Dependents'] = train['Dependents'].replace('3+', 3)
train['Dependents'] = train['Dependents'].astype(float)

New financial features are created to better represent applicant risk:


Total_Income = ApplicantIncome + CoapplicantIncome


Loan_to_Income_Ratio = LoanAmount / Total_Income


EMI_to_Income_Ratio = LoanAmount / Loan_Amount_Term

These engineered features provide stronger financial insights compared to raw values alone and help the model capture applicant repayment capacity more effectively.

In [ ]:
train['Total_Income'] = train['ApplicantIncome'] + train['CoapplicantIncome']

train['Loan_to_Income_Ratio'] = train['LoanAmount'] / train['Total_Income']

train['EMI_to_Income_Ratio'] = train['LoanAmount'] / train['Loan_Amount_Term']

In [ ]:
train["Loan_Status"].isnull().sum()

In [ ]:
train["Loan_Status"].unique()

In [ ]:
from sklearn.feature_selection import mutual_info_classif

# Ensure target is integer type
y_temp = train["Loan_Status"].astype(int)
X_temp = train.drop("Loan_Status", axis=1)

mi_scores = mutual_info_classif(X_temp, y_temp)

mi_df = pd.DataFrame({
    "Feature": X_temp.columns,
    "MI Score": mi_scores
}).sort_values(by="MI Score", ascending=False)

mi_df

In [ ]:
numeric_cols = ["ApplicantIncome", "CoapplicantIncome", "LoanAmount"]

for col in numeric_cols:
    Q1 = train[col].quantile(0.25)
    Q3 = train[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    train[col] = train[col].clip(lower, upper)

The dataset is divided into:

X → Independent variables (applicant features)

Y → Dependent variable (Loan_Status)

Loan_Status is the binary classification target:

1 → Approved

0 → Rejected


In [ ]:
X = train.drop(columns='Loan_Status')
Y = train['Loan_Status']

The dataset is divided into training and validation sets.

This prevents overfitting and ensures that model performance is evaluated on unseen data rather than on the data used for training.

In [ ]:
X_train, X_val, Y_train, Y_val = train_test_split(
    X, Y, test_size=0.2, stratify=Y, random_state=42
)

StandardScaler is applied to normalize feature values.

Scaling is required for algorithms such as Logistic Regression and SVM because they are sensitive to feature magnitude.

Random Forest does not require scaling since it is tree-based.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

Logistic Regression is implemented as a baseline linear classifier.

It provides a reference performance level and helps compare more complex models such as Random Forest and SVM.


In [ ]:
lr = LogisticRegression(random_state=42)
lr.fit(X_train_scaled, Y_train)

lr_pred = lr.predict(X_val_scaled)

print("Logistic Regression Accuracy:",
      accuracy_score(Y_val, lr_pred))

Support Vector Machine is a powerful classification algorithm that identifies the optimal hyperplane separating two classes by maximizing the margin between them.

In this project, SVM with an RBF (Radial Basis Function) kernel was implemented to capture potential non-linear relationships in the dataset.

Feature scaling was applied before training since SVM is sensitive to feature magnitudes.

Although SVM is theoretically strong for structured classification problems, its validation accuracy was slightly lower compared to Random Forest in this dataset. Therefore, it was not selected as the final model.

In [ ]:
svm = SVC(kernel='rbf', probability=True, random_state=42)
svm.fit(X_train_scaled, Y_train)

svm_pred = svm.predict(X_val_scaled)

print("SVM Accuracy:",
      accuracy_score(Y_val, svm_pred))

print("\nClassification Report:\n")
print(classification_report(Y_val, svm_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(Y_val, svm_pred))

print("\nROC-AUC Score:",
      roc_auc_score(Y_val, svm.predict_proba(X_val_scaled)[:,1]))

Random Forest is an ensemble method that builds multiple decision trees and aggregates their predictions.

It captures non-linear relationships and feature interactions, making it well-suited for structured financial datasets.

Based on validation accuracy and ROC-AUC comparison, Random Forest was selected as the final model.

In [ ]:
# Train final model on training split
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, Y_train)

rf_pred = rf.predict(X_val)

print("Accuracy:", accuracy_score(Y_val, rf_pred))

print("\nClassification Report:\n")
print(classification_report(Y_val, rf_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(Y_val, rf_pred))

print("\nROC-AUC Score:",
      roc_auc_score(Y_val, rf.predict_proba(X_val)[:,1]))

**Evaluation Analysis**

The Random Forest model achieved approximately 85% validation accuracy and an ROC-AUC score of approximately 0.84, indicating good class discrimination ability.

*From the confusion matrix:*

The model correctly identified most approved loans (high recall for class 1).

Recall for rejected loans is moderate, indicating some risky applicants may still be approved.



Since the dataset is slightly imbalanced (more approved loans than rejected), accuracy alone is not sufficient. Therefore, precision, recall, F1-score, and ROC-AUC were considered for comprehensive evaluation.

In loan prediction systems, minimizing false approvals (false positives) is important to reduce financial risk.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(Y_val, rf_pred)

plt.figure()
sns.heatmap(cm, annot=True, fmt='d')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Random Forest Confusion Matrix")
plt.show()

testing with an example

In [ ]:
new_person = pd.DataFrame({
    'Gender': [1],
    'Married': [1],
    'Dependents': [1],
    'Education': [1],
    'Self_Employed': [0],
    'ApplicantIncome': [5000],
    'CoapplicantIncome': [2000],
    'LoanAmount': [150],
    'Loan_Amount_Term': [360],
    'Credit_History': [1],
    'Property_Area': [2]
})

In [ ]:
new_person['Total_Income'] = (
    new_person['ApplicantIncome'] +
    new_person['CoapplicantIncome']
)

new_person['Loan_to_Income_Ratio'] = (
    new_person['LoanAmount'] /
    new_person['Total_Income']
)

new_person['EMI_to_Income_Ratio'] = (
    new_person['LoanAmount'] /
    new_person['Loan_Amount_Term']
)

# making them Match training column order
new_person = new_person[X.columns]

In [ ]:
prediction = rf.predict(new_person)
probability = rf.predict_proba(new_person)[:,1]

print("Predicted Status (1=Approved, 0=Rejected):", prediction[0])
print("Approval Probability:", probability[0])

In [ ]:
!pip install streamlit
!npm install -g localtunnel

In [ ]:
!ls

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import pickle

st.set_page_config(page_title="Loan Approval Predictor", page_icon="🏦", layout="wide")

model = pickle.load(open("loan_model.pkl", "rb"))
model_columns = pickle.load(open("model_columns.pkl", "rb"))

# Sidebar
st.sidebar.title("📊 Model Overview")
st.sidebar.write("""
**Model Used:** Random Forest
**Evaluation Metrics:**
- Accuracy
- Precision
- Recall
- F1-Score
- ROC-AUC
""")
st.sidebar.info("This tool predicts whether a loan application will be approved based on applicant information.")

# Title
st.markdown("<h1 style='text-align: center;'>🏦 Loan Approval Prediction System</h1>", unsafe_allow_html=True)
st.markdown("<hr>", unsafe_allow_html=True)

st.subheader("Applicant Details")

col1, col2 = st.columns(2)

with col1:
    gender = st.selectbox("Gender", ["Male", "Female"])
    married = st.selectbox("Married", ["Yes", "No"])
    dependents = st.selectbox("Dependents", [0,1,2,3])
    education = st.selectbox("Education", ["Graduate", "Not Graduate"])
    self_employed = st.selectbox("Self Employed", ["Yes", "No"])

with col2:
    app_income = st.number_input("Applicant Income", min_value=0)
    coapp_income = st.number_input("Coapplicant Income", min_value=0)
    loan_amount = st.number_input("Loan Amount", min_value=0)
    loan_term = st.number_input("Loan Amount Term", min_value=1)
    credit_history = st.selectbox("Credit History", [0,1])
    property_area = st.selectbox("Property Area", ["Rural","Semiurban","Urban"])

st.markdown("---")

if st.button("🔍 Predict Loan Status"):

    data = {
        "Gender": 1 if gender=="Male" else 0,
        "Married": 1 if married=="Yes" else 0,
        "Dependents": dependents,
        "Education": 1 if education=="Graduate" else 0,
        "Self_Employed": 1 if self_employed=="Yes" else 0,
        "ApplicantIncome": app_income,
        "CoapplicantIncome": coapp_income,
        "LoanAmount": loan_amount,
        "Loan_Amount_Term": loan_term,
        "Credit_History": credit_history,
        "Property_Area": {"Rural":0,"Semiurban":1,"Urban":2}[property_area]
    }

    df = pd.DataFrame([data])

    df["Total_Income"] = df["ApplicantIncome"] + df["CoapplicantIncome"]

    if df["Total_Income"].iloc[0] == 0:
        st.error("Total income cannot be zero.")
        st.stop()

    df["Loan_to_Income_Ratio"] = df["LoanAmount"] / df["Total_Income"]
    df["EMI_to_Income_Ratio"] = df["LoanAmount"] / df["Loan_Amount_Term"]

    df = df[model_columns]

    prediction = model.predict(df)[0]
    probability = model.predict_proba(df)[0][1]

    st.markdown("## 📌 Prediction Result")

    colA, colB = st.columns(2)

    with colA:
        if prediction == 1:
            st.success("✅ Loan Approved")
        else:
            st.error("❌ Loan Rejected")

    with colB:
        st.metric("Approval Probability", f"{probability*100:.2f}%")

    st.progress(float(probability))

    # Risk Interpretation
    if probability >= 0.75:
        st.info("Risk Level: Low")
    elif probability >= 0.50:
        st.warning("Risk Level: Moderate")
    else:
        st.error("Risk Level: High")

    st.markdown("---")
    st.subheader("📈 Financial Indicators")
    st.write(f"Total Income: {df['Total_Income'].iloc[0]:,.2f}")
    st.write(f"Loan-to-Income Ratio: {df['Loan_to_Income_Ratio'].iloc[0]:.4f}")
    st.write(f"EMI-to-Income Ratio: {df['EMI_to_Income_Ratio'].iloc[0]:.4f}")

In [ ]:
import pickle
pickle.dump(rf, open("loan_model.pkl", "wb"))
pickle.dump(X.columns.tolist(), open("model_columns.pkl", "wb"))

In [ ]:
!streamlit run app.py &>/dev/null &

In [ ]:
!pip install pyngrok


In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3A7OhrzrnDtRoudXLwKnyosPYKf_5Gbssd2mPhjFyRMm6VaNC")

In [ ]:
public_url = ngrok.connect(8501)
print(public_url)